# AstroProcess: Complete Astrophotography Pipeline

Interactive notebook for processing astronomical images from raw FITS files
to a final color image.

### Supported modes:

| Mode | Filters | Color assignment | Typical use |
|------|---------|-----------------|------------|
| **SHO** (Hubble Palette) | SII, Ha, OIII | S&rarr;R, H&rarr;G, O&rarr;B | Emission nebulae |
| **RGB** (Natural color) | R, G, B | R&rarr;R, G&rarr;G, B&rarr;B | Galaxies, clusters |
| **HaRGB** | R, G, B + Ha | Ha blended into red luminance | Galaxies with HII regions |
| **HaOIII-RGB** | R, G, B + Ha + OIII | Ha&rarr;R, OIII&rarr;B blended with RGB | Mixed objects |

All modes support optional **Luminance (L)** to add detail.

### Features:

- Auto-detection of objects and available filters
- Full calibration (bias, dark, flat, pedestal)
- **Interactive alignment with sliders** (live result)
- **Interactive post-processing with sliders**
- Explanations at every step

---

## Step 1: Install dependencies

- **OpenCV**: alignment and image processing
- **NumPy**: array operations (images are 2D arrays)
- **SciPy**: apply sub-pixel shifts
- **Matplotlib + ipympl**: interactive visualization
- **ipywidgets**: sliders and interactive controls
- **Pillow**: PNG export

In [ ]:
import subprocess, sys

for pkg in ['opencv-python', 'numpy', 'matplotlib', 'Pillow', 'scipy',
            'ipywidgets', 'ipympl']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("Dependencies installed.")

## Step 2: Imports and base functions

### About FITS files

**FITS** (Flexible Image Transport System) is the standard format in astronomy.
Each file has a **header** with metadata (size, data type, capture parameters)
and binary **data** (the pixels) in big-endian format.

Key concepts:

- **BITPIX**: data type (`16` = 16-bit integer, `-32` = float 32-bit)
- **BZERO/BSCALE**: transformation `real_value = BZERO + BSCALE * raw`
- **OFFSET**: camera's electronic pedestal (prevents negative values from noise)

In [ ]:
import numpy as np
import cv2
import re
from scipy.ndimage import shift as ndshift
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import os, glob
from pathlib import Path
from collections import defaultdict

# Try interactive backend for matplotlib
# If it errors out, replace with: %matplotlib inline
%matplotlib widget

# === DIRECTORIES ===
BASE_DIR = Path('.')
LIGHTS_DIR = BASE_DIR / 'lights'
BIAS_DIR   = BASE_DIR / 'biases'
DARKS_DIR  = BASE_DIR / 'darks'
FLATS_DIR  = BASE_DIR / 'flats'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)


def read_fits(filepath):
    """Read a FITS file. Returns (header_dict, data_float32)."""
    with open(str(filepath), 'rb') as f:
        header = {}
        raw_hdr = b''
        end_found = False
        while not end_found:
            block = f.read(2880)
            if len(block) < 2880: break
            raw_hdr += block
            for i in range(0, 2880, 80):
                if block[i:i+3] == b'END': end_found = True; break
        data_start = len(raw_hdr)
        text = raw_hdr.decode('ascii', errors='ignore')
        for i in range(0, len(text), 80):
            card = text[i:i+80]; key = card[:8].strip()
            if key in ('END','COMMENT','HISTORY',''): continue
            if len(card) > 10 and card[8] == '=':
                vs = card[10:].strip()
                if vs.startswith("'"):
                    eq = vs.find("'", 1)
                    header[key] = vs[1:eq].strip() if eq > 0 else vs.strip("'").strip()
                    continue
                if '/' in vs: vs = vs[:vs.index('/')].strip()
                vs = vs.strip()
                if vs in ('T','F'): header[key] = (vs == 'T'); continue
                try: header[key] = float(vs) if ('.' in vs or 'E' in vs.upper()) else int(vs)
                except: header[key] = vs
        bitpix = int(header.get('BITPIX', -32))
        nx, ny = int(header.get('NAXIS1', 0)), int(header.get('NAXIS2', 0))
        bz = float(header.get('BZERO', 0.0))
        bs = float(header.get('BSCALE', 1.0))
        dt = np.dtype({16:'>i2', -32:'>f4', 32:'>i4', -64:'>f8'}[bitpix])
        f.seek(data_start)
        data = np.frombuffer(f.read(nx*ny*dt.itemsize), dtype=dt).reshape((ny, nx))
        data = np.float32(bz) + np.float32(bs) * data.astype(np.float32)
    return header, data


def write_fits(filepath, data):
    """Write float32 data as FITS."""
    fp = str(filepath); ny, nx = data.shape
    cards = ["SIMPLE  =                    T / FITS standard",
             "BITPIX  =                  -32 / float32",
             "NAXIS   =                    2 / 2D image",
             f"NAXIS1  =             {nx:8d} / Width",
             f"NAXIS2  =             {ny:8d} / Height", "END"]
    hdr = b''.join(c.ljust(80).encode('ascii') for c in cards)
    hdr += b' ' * ((2880 - len(hdr) % 2880) % 2880)
    with open(fp, 'wb') as f:
        f.write(hdr); f.write(data.astype('>f4').tobytes())
    print(f"    Saved: {os.path.basename(fp)}")

print("FITS functions ready.")

## Step 3: Automatic directory scanning

The notebook scans the `lights/` folder and automatically detects:

- Which **objects** were photographed (e.g. M97, NGC 2403)
- Which **filters** are available for each object (H, O, S, R, G, B, L)
- How many **frames** per filter
- Which **mode** is recommended (SHO or RGB)

The filename follows the format: `date_time_FILTER_OBJECT_exposure_number.fits`

In [ ]:
# Parse filenames
# Format: 2026-02-24_01-18-04_R_NGC 2403_240.00s_0000.fits
# The filter is always a single uppercase letter at position 3 of the name

pattern = re.compile(
    r'^(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})_'  # date_time
    r'([A-Z])_'                                     # filter
    r'(.+?)_'                                       # object (may contain spaces)
    r'(\d+\.\d+s)_'                                 # exposure
    r'(\d+)\.fits$'                                  # number
)

# Scan lights
catalog = defaultdict(lambda: defaultdict(list))  # catalog[object][filter] = [files]

for fn in sorted(os.listdir(str(LIGHTS_DIR))):
    if not fn.endswith('.fits'):
        continue
    m = pattern.match(fn)
    if m:
        timestamp, filt, obj, exp, num = m.groups()
        catalog[obj][filt].append({
            'file': str(LIGHTS_DIR / fn),
            'exp': float(exp.replace('s', '')),
            'name': fn
        })
    else:
        # Fallback: try to detect filter from the usual position
        parts = fn.replace('.fits', '').split('_')
        if len(parts) >= 5:
            filt = parts[2]
            obj = parts[3]
            catalog[obj][filt].append({
                'file': str(LIGHTS_DIR / fn),
                'exp': 0,
                'name': fn
            })

# Show summary
print("=" * 60)
print("  DETECTED OBJECTS")
print("=" * 60)

for obj in sorted(catalog.keys()):
    filters = catalog[obj]
    has_sho = all(f in filters for f in ['H', 'O', 'S'])
    has_rgb = all(f in filters for f in ['R', 'G', 'B'])
    has_nb = any(f in filters for f in ['H', 'O', 'S'])
    has_L = 'L' in filters

    print(f"\n  {obj}:")
    for filt in sorted(filters.keys()):
        frames = filters[filt]
        exps = [f['exp'] for f in frames]
        print(f"    {filt}: {len(frames)} frames, exp={exps[0]:.0f}s")

    # Show ALL possible modes
    modes = []
    if has_sho:
        modes.append("SHO")
    if has_rgb:
        modes.append("RGB")
    if has_rgb and 'H' in filters:
        modes.append("HaRGB")
    if has_rgb and 'H' in filters and 'O' in filters:
        modes.append("HaOIII-RGB")
    if has_L and not has_rgb and not has_sho:
        modes.append("Luminance (mono)")
    if not modes:
        modes.append("partial")

    modes_str = ", ".join(modes)
    print(f"    Possible modes: {modes_str}" + (" + L" if has_L and has_rgb or has_sho else ""))

print("\n" + "=" * 60)

## Step 4: Select object and mode

Use the **dropdowns** to select the object and composition mode.
When you change the object, the modes update automatically.

| Mode | Required filters | Result | Ideal for |
|------|-----------------|--------|-----------|
| **SHO** | Ha + OIII + SII | False color (Hubble Palette) | Emission nebulae |
| **RGB** | R + G + B | Natural color | Galaxies, clusters |
| **HaRGB** | R + G + B + Ha | Natural color + Ha detail | Galaxies with HII regions |
| **HaOIII-RGB** | R + G + B + Ha + OIII | RGB enriched with NB | Mixed objects |

### What is HaRGB?

When you have **RGB + Ha**, you can blend Ha into the red channel to highlight
ionized hydrogen regions (nebulae, HII regions in galaxies).
The result is natural color but with much more visible nebulosity.

### What is HaOIII-RGB?

Similar to HaRGB but also blends OIII into the blue channel.
Highlights both hydrogen (red) and oxygen (blue) regions.

### Luminance mode (mono)

If you only have luminance (L) frames, the notebook generates a high-quality
**monochromatic** image. It is calibrated, aligned, stacked and stretched
like any other channel. The result is a grayscale image.

In [ ]:
# ==========================================================
# INTERACTIVE SELECTION - Use the dropdowns to choose
# ==========================================================

def get_modes_for_object(obj_name):
    """Calculate possible modes for a given object."""
    filts = sorted(catalog[obj_name].keys())
    _has_sho = all(f in filts for f in ['H', 'O', 'S'])
    _has_rgb = all(f in filts for f in ['R', 'G', 'B'])
    _has_Ha = 'H' in filts
    _has_OIII = 'O' in filts
    _has_L = 'L' in filts
    modes = []
    if _has_sho: modes.append('SHO')
    if _has_rgb: modes.append('RGB')
    if _has_rgb and _has_Ha: modes.append('HaRGB')
    if _has_rgb and _has_Ha and _has_OIII: modes.append('HaOIII-RGB')
    if _has_L: modes.append('Lum')
    if not modes: modes.append('partial')
    return modes

def auto_mode_for_object(obj_name):
    """Auto-detect the best mode for an object."""
    filts = sorted(catalog[obj_name].keys())
    if all(f in filts for f in ['H', 'O', 'S']): return 'SHO'
    if all(f in filts for f in ['R', 'G', 'B']):
        if 'H' in filts and 'O' in filts: return 'HaOIII-RGB'
        if 'H' in filts: return 'HaRGB'
        return 'RGB'
    if 'L' in filts: return 'Lum'
    return 'partial'

# --- Dropdown: Object ---
object_names = sorted(catalog.keys())
dd_object = widgets.Dropdown(
    options=object_names,
    value=object_names[0],
    description='Object:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

# --- Dropdown: Mode ---
initial_modes = get_modes_for_object(object_names[0])
dd_mode = widgets.Dropdown(
    options=initial_modes,
    value=auto_mode_for_object(object_names[0]),
    description='Mode:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='350px')
)

out_selection = widgets.Output()

def on_object_change(change):
    """When the object changes, update available modes."""
    obj = change['new']
    new_modes = get_modes_for_object(obj)
    dd_mode.options = new_modes
    dd_mode.value = auto_mode_for_object(obj)
    update_selection_info()

def update_selection_info():
    obj = dd_object.value
    mode = dd_mode.value
    filts = sorted(catalog[obj].keys())
    with out_selection:
        clear_output(wait=True)
        print(f"  Object:    {obj}")
        print(f"  Filters:   {filts}")
        for f in sorted(catalog[obj].keys()):
            frames = catalog[obj][f]
            exps = set(fr['exp'] for fr in frames)
            exp_str = ', '.join(f'{e:.0f}s' for e in sorted(exps))
            print(f"    {f}: {len(frames)} frames, exp={exp_str}")
        print(f"  Mode:      {mode}")

dd_object.observe(on_object_change, names='value')
dd_mode.observe(lambda c: update_selection_info(), names='value')

display(widgets.VBox([
    widgets.HTML('<h3>Object and mode selection</h3>'
                 '<p>1. Select the object and composition mode.<br>'
                 '2. When ready, <b>run the next cell</b> to confirm.</p>'),
    widgets.HBox([dd_object, widgets.HTML('&nbsp;&nbsp;'), dd_mode]),
]))
display(out_selection)
update_selection_info()

### Confirm selection

After choosing the object and mode in the dropdowns above,
**run this cell** to confirm the selection and continue.

In [ ]:
# ==========================================================
# Read selection from dropdowns
# ==========================================================
OBJECT = dd_object.value
MODE = dd_mode.value

# Detect available filters
filters_available = sorted(catalog[OBJECT].keys())
has_sho = all(f in filters_available for f in ['H', 'O', 'S'])
has_rgb = all(f in filters_available for f in ['R', 'G', 'B'])
has_Ha = 'H' in filters_available
has_OIII = 'O' in filters_available
has_L = 'L' in filters_available

# List all possible modes for this object
possible_modes = get_modes_for_object(OBJECT)

print(f"Object:          {OBJECT}")
print(f"Filters:         {filters_available}")
print(f"Possible modes:  {possible_modes}")
print(f"Selected mode:   {MODE}")
print()

# Determine which filters to process based on mode
if MODE == 'SHO':
    color_filters = ['H', 'O', 'S']
    nb_mix_filters = []
elif MODE == 'HaRGB':
    color_filters = ['R', 'G', 'B']
    nb_mix_filters = ['H']
elif MODE == 'HaOIII-RGB':
    color_filters = ['R', 'G', 'B']
    nb_mix_filters = ['H', 'O']
elif MODE == 'Lum':
    color_filters = ['L']
    nb_mix_filters = []
else:  # RGB
    color_filters = ['R', 'G', 'B']
    nb_mix_filters = []

# Add L as extra luminance channel (except in Lum mode where it's already the primary)
extra_L = has_L and MODE != 'Lum' and 'L' not in color_filters
all_filters = list(dict.fromkeys(color_filters + nb_mix_filters + (['L'] if extra_L else [])))

print(f"Color filters:       {color_filters}")
if nb_mix_filters:
    print(f"NB filters to blend: {nb_mix_filters}")
print(f"Total to process:    {all_filters}")

if MODE not in possible_modes and possible_modes:
    print(f"\nWARNING: mode '{MODE}' is not possible with these filters.")
    print(f"Valid modes: {possible_modes}")

## Step 5: Pedestal (OFFSET) diagnostic

### What is the OFFSET?

The camera adds an **electronic pedestal** to all pixels to prevent
read noise from producing negative values. If the calibration masters
(bias, dark, flat) were captured with a different OFFSET than the lights,
the difference must be compensated.

We measure the median of the bias (high OFFSET) and of a light (low OFFSET)
to calculate the required correction.

In [ ]:
# Read bias
_, bias_data = read_fits(BIAS_DIR / 'mbias.fits')
bias_med = np.median(bias_data)

# Read a sample light
sample_file = catalog[OBJECT][color_filters[0]][0]['file']
_, sample_light = read_fits(sample_file)
light_med = np.median(sample_light)

PEDESTAL = bias_med - light_med

print(f"  Bias median:     {bias_med:.1f} ADU")
print(f"  Light median:    {light_med:.1f} ADU")
print(f"  Pedestal:        {PEDESTAL:.1f} ADU")

# Check available darks
dark_files = sorted(glob.glob(str(DARKS_DIR / 'mdarks_*.fits')))
darks = {}
dark_exps = []
for df in dark_files:
    exp_str = os.path.basename(df).replace('mdarks_', '').replace('s.fits', '')
    try:
        exp = float(exp_str)
        _, darks[exp] = read_fits(df)
        dark_exps.append(exp)
        print(f"  Dark {exp:.0f}s: median={np.median(darks[exp]):.1f}")
    except: pass

# Show dark matching for each filter
print(f"\n  Available darks: {sorted(dark_exps)}s")
print(f"  Checking match with lights:")
for filt in all_filters:
    if filt not in catalog[OBJECT]: continue
    light_exp = catalog[OBJECT][filt][0]['exp']
    if light_exp == 0: continue
    best_exp = min(dark_exps, key=lambda d: abs(d - light_exp)) if dark_exps else 0
    diff = abs(best_exp - light_exp)
    if diff == 0:
        status = "EXACT"
    elif diff <= 30:
        status = f"close (diff={diff:.0f}s) -> will be scaled"
    else:
        status = f"VERY DIFFERENT (diff={diff:.0f}s) -> will be scaled proportionally"
    n_frames = len(catalog[OBJECT][filt])
    hot_warn = " [1 frame: hot pixels will be corrected with median filter]" if n_frames == 1 else ""
    print(f"    {filt} ({light_exp:.0f}s, {n_frames}fr) -> dark {best_exp:.0f}s: {status}{hot_warn}")

# Available flats
flats = {}
for filt in all_filters:
    fp = FLATS_DIR / f'mflat_{filt}.fits'
    if fp.exists():
        _, flats[filt] = read_fits(fp)
print(f"\n  Flats: {list(flats.keys())}")

## Step 6: Calibration

### Formula:

```
calibrated = (light - scaled_dark) / flat
```

| Term | Corrects | Obtained from |
|------|----------|---------------|
| **Dark** | Thermal noise + fixed pattern | Photo with shutter closed, same exposure |
| **Pedestal** | OFFSET difference between masters and lights | `median(bias) - median(light)` |
| **Flat** | Vignetting + dust on sensor | Photo of uniform field, normalized (mean=1) |

### Dark scaling

Dark current scales **linearly** with exposure time.
If we don't have a dark with exactly the same exposure as the light, we extract
the pure thermal component `(dark - bias)` and scale it proportionally:

```
scaled_dark = bias + (dark - bias) * (light_exp / dark_exp) - pedestal
```

This is critical: a 300s dark used without scaling on a 400s light leaves ~25%
of hot pixels uncorrected.

### Cosmetic correction (hot pixels)

**Hot pixels** are defective pixels that accumulate extra charge.
With 3+ frames, median stacking removes them automatically.
But with **only 1 frame** (like Ha or OIII sometimes), we need to detect them
by comparing each pixel with its neighbors and replacing outliers.

In [ ]:
from scipy.ndimage import median_filter

def scale_dark(dark, bias, light_exp, dark_exp, pedestal):
    """Scale a dark to match the light's exposure time.

    Dark current is linear with time:
      scaled_dark = bias + (dark - bias) * (light_exp / dark_exp) - pedestal

    This extracts only the thermal component, scales it, and compensates the pedestal.
    """
    thermal = dark - bias  # pure thermal component
    scale = light_exp / dark_exp if dark_exp > 0 else 1.0
    return bias + thermal * scale - pedestal

def cosmetic_correction(data, sigma=5.0, size=5):
    """Detect and correct hot/cold pixels by comparing with local median.

    A pixel is considered 'hot' if it differs from the median of its neighbors
    by more than sigma * local_standard_deviation. It is replaced by the median.
    Essential for single frames where stacking cannot remove them.
    """
    med = median_filter(data, size=size)
    diff = np.abs(data - med)
    # Robust local noise estimation
    mad = median_filter(diff, size=size * 2 + 1)
    threshold = sigma * np.clip(mad, 1.0, None)  # avoid division by 0
    mask = diff > threshold
    corrected = data.copy()
    corrected[mask] = med[mask]
    n_fixed = np.sum(mask)
    return corrected, n_fixed

def calibrate(light, dark_scaled, flat):
    """Calibrate a frame: (light - scaled_dark) / flat"""
    return (light - dark_scaled) / np.clip(flat, 0.3, 3.0)

def get_best_dark(exp, darks_dict):
    """Select the dark with the closest exposure."""
    if not darks_dict:
        return None, 0
    best_exp = min(darks_dict.keys(), key=lambda d: abs(d - exp))
    return darks_dict[best_exp], best_exp

print(f"Calibrating {OBJECT} ({MODE})...\n")
calibrated = defaultdict(list)      # calibrated[filter] = [array, ...]
calibrated_exps = defaultdict(list)  # calibrated_exps[filter] = [exp_seconds, ...]

for filt in all_filters:
    if filt not in catalog[OBJECT]:
        print(f"  {filt}: no frames, skipping")
        continue
    flat = flats.get(filt, np.ones_like(bias_data))
    n_frames = len(catalog[OBJECT][filt])
    for frame_info in catalog[OBJECT][filt]:
        fp = frame_info['file']
        exp = frame_info['exp']
        hdr, light = read_fits(fp)
        # If exp is 0 (could not parse), try from header
        if exp == 0:
            exp = float(hdr.get('EXPTIME', 300))
        dark_raw, dark_exp = get_best_dark(exp, darks)
        if dark_raw is None:
            print(f"  WARNING: no dark available, skipping {os.path.basename(fp)}")
            continue
        # Scale dark to the correct exposure
        dark_scaled = scale_dark(dark_raw, bias_data, exp, dark_exp, PEDESTAL)
        if abs(dark_exp - exp) > 1:
            scale_info = f" (dark {dark_exp:.0f}s scaled to {exp:.0f}s)"
        else:
            scale_info = ""
        cal = calibrate(light, dark_scaled, flat)
        # Cosmetic correction (hot pixels) - especially critical for single frames
        cal, n_hot = cosmetic_correction(cal, sigma=5.0, size=5)
        calibrated[filt].append(cal)
        calibrated_exps[filt].append(exp)
        hot_info = f", {n_hot} hot px corrected" if n_hot > 0 else ""
        print(f"  {filt} {os.path.basename(fp)}: med={np.median(cal):.1f}{scale_info}{hot_info}")

print("\nCalibration complete.")
print("Calibrated filters:", {f: len(calibrated[f]) for f in calibrated})

## Step 7: Intra-filter alignment + Stacking

### Intra-filter alignment

Before stacking, we align frames **within** each filter.
Between exposures the telescope moves slightly.
We use **phase correlation** (FFT) to detect the shift.

### Median stacking

The **median** is robust against outliers (cosmic rays, satellites, hot pixels).
With 3 frames: if a pixel has values [100, 102, 50000], the median is 102.

### Mixed exposures

If a filter has frames with different exposure times (e.g. 30s, 60s, 120s),
the notebook **normalizes them to ADU/second** before aligning and stacking.
This prevents phase correlation from failing due to brightness differences.

> If stacks already exist in `output/`, they are reused. Set `FORCE_RESTACK = True` to regenerate.

In [ ]:
FORCE_RESTACK = False  # True to re-stack even if files already exist

def phase_align(ref, target, label=""):
    """Align target to ref using phase correlation."""
    h, w = ref.shape
    def norm(img):
        p1, p99 = np.percentile(img, [1, 99.5])
        return np.clip((img - p1) / max(p99 - p1, 1), 0, 1).astype(np.float64)
    ref_n, tgt_n = norm(ref), norm(target)
    # Downscale for speed
    s = 0.125
    r_s = cv2.resize(ref_n, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    t_s = cv2.resize(tgt_n, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    hs, ws = r_s.shape
    hann = cv2.createHanningWindow((ws, hs), cv2.CV_64F)
    (dx_s, dy_s), conf = cv2.phaseCorrelate(r_s * hann, t_s * hann)
    dx, dy = dx_s / s, dy_s / s
    if abs(dx) > 500 or abs(dy) > 500 or conf < 0.005:
        if label: print(f"    {label}: shift discarded (dx={dx:.0f}, dy={dy:.0f}, conf={conf:.3f})")
        return target
    if label: print(f"    {label}: dx={dx:+.1f}, dy={dy:+.1f}, conf={conf:.3f}")
    return ndshift(target, [dy, dx], order=1, mode='constant', cval=float(np.median(target)))

# Prefix for stacks (includes object to avoid mixing)
obj_tag = OBJECT.replace(' ', '_')

# Align + stack
print(f"Aligning and stacking {OBJECT}...\n")
stacked = {}

for filt in all_filters:
    stack_file = OUTPUT_DIR / f'stacked_{obj_tag}_{filt}.fits'

    if stack_file.exists() and not FORCE_RESTACK:
        _, stacked[filt] = read_fits(stack_file)
        print(f"  {filt}: REUSED ({stack_file.name})")
        continue

    frames = calibrated.get(filt, [])
    exps = calibrated_exps.get(filt, [])
    if not frames:
        continue

    # If there are different exposure times, normalize to ADU/second
    # before aligning and stacking. This is critical for objects like clusters
    # where frames of different durations are captured.
    unique_exps = set(exps)
    if len(unique_exps) > 1 and all(e > 0 for e in exps):
        print(f"  {filt}: mixed exposures ({sorted(unique_exps)}s) -> normalizing to ADU/s")
        frames = [f / e for f, e in zip(frames, exps)]

    # Intra-filter alignment
    if len(frames) > 1:
        print(f"  {filt}: aligning {len(frames)} frames...")
        aligned_f = [frames[0]]
        for i in range(1, len(frames)):
            aligned_f.append(phase_align(frames[0], frames[i], f"frame {i}"))
        stack = np.median(np.array(aligned_f), axis=0)
    else:
        stack = frames[0]

    stacked[filt] = stack
    write_fits(str(stack_file), stack)
    print(f"  {filt}: stacked (med={np.median(stack):.1f}, P99.9={np.percentile(stack, 99.9):.0f})")

# Visualize
n = len(stacked)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4))
if n == 1: axes = [axes]
for ax, (f, d) in zip(axes, stacked.items()):
    v1, v2 = np.percentile(d, [1, 99.5])
    ax.imshow(d, cmap='gray', vmin=v1, vmax=v2, origin='lower')
    ax.set_title(f'Stacked {f}', fontsize=11); ax.axis('off')
plt.suptitle(f'{OBJECT} - Stacked channels', fontsize=13)
plt.tight_layout(); plt.show()

print(f"\nStacks ready: {list(stacked.keys())}")

## Step 8: Inter-filter alignment (interactive)

Channels from different filters may be shifted relative to each other
(filter change, differential flexure).

### How to use the sliders?

1. Look at the **zoom** (right): stars should appear **white and round**
2. If you see color fringes (red/green/blue separated), adjust the `dx` and `dy` sliders
3. Use the **Zoom X/Y** controls to navigate to different stars
4. When satisfied, run the **next cell** to apply at full resolution

The initial values come from auto-detection. Sometimes they only need fine-tuning.

**Important**: In HaRGB and HaOIII-RGB modes, the preview shows the actual
narrowband blend, so you can see if Ha or OIII are misaligned relative to the RGB.

In [ ]:
# Prepare previews (1/8 scale for speed)
PREVIEW_SCALE = 0.125

def quick_stretch(data):
    p1, p999 = np.percentile(data, [1, 99.9])
    d = np.clip((data - p1) / max(p999 - p1, 1), 0, 1)
    return np.power(d, 0.3)

previews = {}
for filt in stacked:
    small = cv2.resize(stacked[filt].astype(np.float32), None,
                       fx=PREVIEW_SCALE, fy=PREVIEW_SCALE, interpolation=cv2.INTER_AREA)
    previews[filt] = quick_stretch(small)

h_prev, w_prev = list(previews.values())[0].shape
ref_filt = color_filters[0]  # H for SHO, R for RGB

# Auto-detect initial shifts
auto_shifts = {}
for filt in all_filters:
    if filt == ref_filt or filt not in stacked:
        continue
    ref_s = cv2.resize(stacked[ref_filt].astype(np.float32), None,
                       fx=PREVIEW_SCALE, fy=PREVIEW_SCALE, interpolation=cv2.INTER_AREA)
    tgt_s = cv2.resize(stacked[filt].astype(np.float32), None,
                       fx=PREVIEW_SCALE, fy=PREVIEW_SCALE, interpolation=cv2.INTER_AREA)
    def norm64(img):
        p1, p99 = np.percentile(img, [1, 99.5])
        return np.clip((img - p1) / max(p99 - p1, 1), 0, 1).astype(np.float64)
    r_n, t_n = norm64(ref_s), norm64(tgt_s)
    hs, ws = r_n.shape
    hann = cv2.createHanningWindow((ws, hs), cv2.CV_64F)
    (dx_s, dy_s), conf = cv2.phaseCorrelate(r_n * hann, t_n * hann)
    dx, dy = dx_s / PREVIEW_SCALE, dy_s / PREVIEW_SCALE
    if abs(dx) < 100 and abs(dy) < 100 and conf > 0.01:
        auto_shifts[filt] = (round(dx, 1), round(dy, 1))
    else:
        auto_shifts[filt] = (0.0, 0.0)
    print(f"  {filt}->{ref_filt}: dx={dx:+.1f}, dy={dy:+.1f} (conf={conf:.3f})"
          f"{' [auto]' if filt in auto_shifts and auto_shifts[filt] != (0,0) else ''}")

# Create sliders
sliders = {}
filters_to_align = [f for f in all_filters if f != ref_filt and f in stacked]

for filt in filters_to_align:
    init_dx, init_dy = auto_shifts.get(filt, (0, 0))
    sliders[f'{filt}_dx'] = widgets.FloatSlider(
        value=init_dx, min=-60, max=60, step=0.5,
        description=f'{filt} dx:', continuous_update=True,
        style={'description_width': '50px'}, layout=widgets.Layout(width='380px'))
    sliders[f'{filt}_dy'] = widgets.FloatSlider(
        value=init_dy, min=-60, max=60, step=0.5,
        description=f'{filt} dy:', continuous_update=True,
        style={'description_width': '50px'}, layout=widgets.Layout(width='380px'))

zoom_x = widgets.IntSlider(value=w_prev//2, min=80, max=w_prev-80, step=5,
    description='Zoom X:', style={'description_width': '60px'},
    layout=widgets.Layout(width='380px'))
zoom_y = widgets.IntSlider(value=h_prev//2, min=80, max=h_prev-80, step=5,
    description='Zoom Y:', style={'description_width': '60px'},
    layout=widgets.Layout(width='380px'))
zoom_size = widgets.IntSlider(value=80, min=30, max=200, step=10,
    description='Size:', style={'description_width': '60px'},
    layout=widgets.Layout(width='380px'))

out_align = widgets.Output()

def update_alignment(**kwargs):
    with out_align:
        clear_output(wait=True)
        shifted = {}
        shifted[ref_filt] = previews[ref_filt]
        for filt in filters_to_align:
            dx = sliders.get(f'{filt}_dx')
            dy = sliders.get(f'{filt}_dy')
            if dx is None:
                shifted[filt] = previews.get(filt, np.zeros_like(previews[ref_filt]))
                continue
            dx_px = dx.value * PREVIEW_SCALE
            dy_px = dy.value * PREVIEW_SCALE
            if abs(dx_px) < 0.05 and abs(dy_px) < 0.05:
                shifted[filt] = previews[filt]
            else:
                shifted[filt] = ndshift(previews[filt], [dy_px, dx_px],
                                        order=1, mode='constant', cval=0)
        # Compose RGB according to mode (including NB blend to see actual effect)
        zeros = np.zeros((h_prev, w_prev))
        if MODE == 'SHO':
            r_ch = shifted.get('S', zeros)
            g_ch = shifted.get('H', zeros)
            b_ch = shifted.get('O', zeros)
        elif MODE == 'HaRGB':
            r_rgb = shifted.get('R', zeros)
            ha_ch = shifted.get('H', zeros)
            r_ch = np.clip(0.65 * r_rgb + 0.35 * ha_ch, 0, 1)
            g_ch = shifted.get('G', zeros)
            b_ch = shifted.get('B', zeros)
        elif MODE == 'HaOIII-RGB':
            r_rgb = shifted.get('R', zeros)
            ha_ch = shifted.get('H', zeros)
            b_rgb = shifted.get('B', zeros)
            oiii_ch = shifted.get('O', zeros)
            r_ch = np.clip(0.65 * r_rgb + 0.35 * ha_ch, 0, 1)
            g_ch = shifted.get('G', zeros)
            b_ch = np.clip(0.65 * b_rgb + 0.35 * oiii_ch, 0, 1)
        else:
            r_ch = shifted.get('R', zeros)
            g_ch = shifted.get('G', zeros)
            b_ch = shifted.get('B', zeros)
        rgb = np.clip(np.stack([r_ch, g_ch, b_ch], axis=-1), 0, 1).astype(np.float32)
        cx, cy, zs = zoom_x.value, zoom_y.value, zoom_size.value
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        axes[0].imshow(rgb, origin='lower')
        rect = plt.Rectangle((cx-zs, cy-zs), 2*zs, 2*zs, lw=2, ec='yellow', fc='none')
        axes[0].add_patch(rect)
        axes[0].set_title(f'{OBJECT} - {MODE}', fontsize=11); axes[0].axis('off')
        y1, y2 = max(0, cy-zs), min(h_prev, cy+zs)
        x1, x2 = max(0, cx-zs), min(w_prev, cx+zs)
        axes[1].imshow(rgb[y1:y2, x1:x2], origin='lower', interpolation='nearest')
        axes[1].set_title('Zoom (stars = white, no fringes)', fontsize=11); axes[1].axis('off')
        plt.tight_layout(); plt.show()

for ctrl in list(sliders.values()) + [zoom_x, zoom_y, zoom_size]:
    ctrl.observe(lambda c: update_alignment(), names='value')

filter_boxes = []
for filt in filters_to_align:
    if f'{filt}_dx' in sliders:
        name_map = {'O':'OIII','S':'SII','H':'Ha','R':'Red','G':'Green','B':'Blue','L':'Lum'}
        filter_boxes.append(widgets.VBox([
            widgets.HTML(f'<b>{filt} ({name_map.get(filt, filt)})</b>'),
            sliders[f'{filt}_dx'], sliders[f'{filt}_dy']]))

display(widgets.VBox([
    widgets.HTML(f'<h3>Inter-filter alignment: {MODE} ({OBJECT})</h3>'
                 f'<p>Reference: {ref_filt}. Move sliders until stars are white.</p>'),
    widgets.HBox(filter_boxes),
    widgets.HTML('<br>'),
    widgets.VBox([widgets.HTML('<b>Zoom navigation</b>'), zoom_x, zoom_y, zoom_size]),
]))
display(out_align)
update_alignment()

## Step 9: Apply alignment at full resolution

Once satisfied with the sliders, run this cell.
It applies the same shifts to the full-resolution images.

In [ ]:
print("Applying alignment at full resolution...\n")
aligned = {ref_filt: stacked[ref_filt]}

for filt in filters_to_align:
    if filt not in stacked:
        continue
    dx_s = sliders.get(f'{filt}_dx')
    dy_s = sliders.get(f'{filt}_dy')
    dx = dx_s.value if dx_s else 0
    dy = dy_s.value if dy_s else 0
    if abs(dx) < 0.1 and abs(dy) < 0.1:
        aligned[filt] = stacked[filt]
        print(f"  {filt}: no shift")
    else:
        print(f"  {filt}: ({dy:+.1f}, {dx:+.1f})...", end=" ", flush=True)
        aligned[filt] = ndshift(stacked[filt], [dy, dx], order=1,
                                mode='constant', cval=float(np.median(stacked[filt])))
        print("OK")

print("\nAlignment applied.")

## Step 10: Color composition

### Composition modes:

- **SHO**: SII&rarr;R, Ha&rarr;G, OIII&rarr;B (false color, reveals chemistry)
- **RGB**: R&rarr;R, G&rarr;G, B&rarr;B (natural color)
- **HaRGB**: RGB + Ha blended into red channel (highlights HII regions)
- **HaOIII-RGB**: RGB + Ha in red + OIII in blue (highlights emission)

### How does the NB blend work in HaRGB / HaOIII-RGB?

The `NB_MIX` parameter controls how much narrowband is blended:

```
R_final = (1 - NB_MIX) * R  +  NB_MIX * Ha
B_final = (1 - NB_MIX) * B  +  NB_MIX * OIII   (HaOIII-RGB only)
```

| NB_MIX | Effect |
|--------|--------|
| 0.0 | Pure RGB (ignores narrowband) |
| 0.3 | Soft blend (recommended) |
| 0.5 | Medium blend |
| 0.7+ | NB dominant (may look artificial) |

### Stretch by mode:

| Parameter | SHO | RGB/HaRGB | Effect |
|-----------|-----|-----------|--------|
| `STRETCH` | 0.05-0.1 | 0.2-0.5 | Lower = more aggressive |
| `GAMMA` | 3.0-4.0 | 2.0-3.0 | Higher = brighter midtones |
| `WHITE_PCT` | 99.95 | 99.9 | White point percentile |

In [ ]:
def prepare_channel(data, white_pct=99.95):
    clean = np.clip(data, 0, None)
    white = np.percentile(clean, white_pct)
    if white <= 0: return np.zeros_like(data)
    return np.clip(clean / white, 0, 1)

def asinh_stretch(data, strength=0.08):
    factor = 1.0 / max(strength, 0.001)
    return np.arcsinh(data * factor) / np.arcsinh(factor)

def gamma_corr(data, gamma=3.5):
    return np.power(np.clip(data, 0, 1), 1.0 / gamma)

# =============================================
# Defaults per mode
# =============================================
if MODE == 'SHO':
    STRETCH   = 0.08
    GAMMA     = 3.5
    WHITE_PCT = 99.95
    NB_MIX    = 0     # Not applicable in SHO
else:
    STRETCH   = 0.3
    GAMMA     = 2.5
    WHITE_PCT = 99.9
    NB_MIX    = 0.35  # How much NB to blend in HaRGB/HaOIII-RGB
# =============================================

print(f"Mode: {MODE}")
print(f"STRETCH={STRETCH}, GAMMA={GAMMA}, WHITE_PCT={WHITE_PCT}")
if MODE in ('HaRGB', 'HaOIII-RGB'):
    print(f"NB_MIX={NB_MIX} (narrowband blend into RGB)")
print()

def process_ch(data):
    return gamma_corr(asinh_stretch(prepare_channel(data, WHITE_PCT), STRETCH), GAMMA)

# --- Composition by mode ---
if MODE == 'SHO':
    r = process_ch(aligned['S'])
    g = process_ch(aligned['H'])
    b = process_ch(aligned['O'])
    labels = ['SII (Red)', 'Ha (Green)', 'OIII (Blue)']

elif MODE == 'HaRGB':
    r_rgb = process_ch(aligned['R'])
    g = process_ch(aligned['G'])
    b = process_ch(aligned['B'])
    ha = process_ch(aligned['H'])
    # Blend Ha into red channel
    r = np.clip((1 - NB_MIX) * r_rgb + NB_MIX * ha, 0, 1)
    labels = [f'R + Ha (mix={NB_MIX})', 'G (Green)', 'B (Blue)']
    print(f"  Ha blended into Red with weight {NB_MIX}")

elif MODE == 'HaOIII-RGB':
    r_rgb = process_ch(aligned['R'])
    g = process_ch(aligned['G'])
    b_rgb = process_ch(aligned['B'])
    ha = process_ch(aligned['H'])
    oiii = process_ch(aligned['O'])
    # Blend Ha into red, OIII into blue
    r = np.clip((1 - NB_MIX) * r_rgb + NB_MIX * ha, 0, 1)
    b = np.clip((1 - NB_MIX) * b_rgb + NB_MIX * oiii, 0, 1)
    labels = [f'R + Ha (mix={NB_MIX})', 'G (Green)', f'B + OIII (mix={NB_MIX})']
    print(f"  Ha blended into Red, OIII blended into Blue (weight {NB_MIX})")

elif MODE == 'Lum':
    # Monochromatic mode: luminance only
    lum = process_ch(aligned['L'])
    r, g, b = lum, lum, lum
    labels = ['Luminance', '', '']
    print("  Luminance mode: monochromatic image")

else:  # RGB
    r = process_ch(aligned['R'])
    g = process_ch(aligned['G'])
    b = process_ch(aligned['B'])
    labels = ['R (Red)', 'G (Green)', 'B (Blue)']

composed = np.clip(np.stack([r, g, b], axis=-1), 0, 1).astype(np.float32)

if MODE == 'Lum':
    # For Lum mode, show only the grayscale image
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(lum, cmap='gray', vmin=0, vmax=1, origin='lower')
    ax.set_title(f'{OBJECT} - Luminance', fontsize=14); ax.axis('off')
    plt.tight_layout(); plt.show()
    print(f"  Luminance: mean={lum.mean():.3f}, median={np.median(lum):.3f}")
else:
    for name, ch in zip(labels, [r, g, b]):
        if name:
            print(f"  {name}: mean={ch.mean():.3f}, median={np.median(ch):.3f}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    cmaps = ['Reds', 'Greens', 'Blues']
    for ax, (name, ch, cm) in zip(axes.flat[:3], zip(labels, [r, g, b], cmaps)):
        ax.imshow(ch, cmap=cm, vmin=0, vmax=1, origin='lower')
        ax.set_title(name, fontsize=12); ax.axis('off')
    axes[1,1].imshow(composed, origin='lower')
    axes[1,1].set_title(f'{MODE} Combined', fontsize=12); axes[1,1].axis('off')
    plt.suptitle(f'{OBJECT} - {MODE}', fontsize=14)
    plt.tight_layout(); plt.show()

## Step 11: Luminance (LRGB) - Optional

Luminance adds **detail and sharpness** (especially in stars)
without changing the colors.

| LUM_WEIGHT | Effect |
|------------|--------|
| 0.0 | No luminance (pure color only) |
| 0.2-0.3 | Soft blend (adds detail, preserves color) |
| 0.5 | Medium blend |
| 1.0 | Luminance only (loses color, NOT recommended) |

With few L frames (1-2), use 0.0 or 0.2 maximum.

In [ ]:
LUM_WEIGHT = 0.3  # Adjust based on amount of L data

if MODE == 'Lum':
    final_pre = composed.copy()
    print("Luminance mode: LRGB does not apply.")
elif 'L' in aligned and LUM_WEIGHT > 0:
    l_ch = process_ch(aligned['L'])
    sho_8 = (composed * 255).astype(np.uint8)
    hsv = cv2.cvtColor(sho_8, cv2.COLOR_RGB2HSV).astype(np.float32)
    v_sho = hsv[:,:,2] / 255.0
    hsv[:,:,2] = np.clip((LUM_WEIGHT * l_ch + (1 - LUM_WEIGHT) * v_sho) * 255, 0, 255)
    final_pre = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32) / 255.0
    print(f"LRGB applied (weight={LUM_WEIGHT})")
else:
    final_pre = composed.copy()
    print(f"No luminance (LUM_WEIGHT={LUM_WEIGHT})")

## Step 12: Interactive post-processing

Use the sliders to adjust color, saturation and brightness **in real time**.

### Parameter guide:

| Slider | What it does | Typical values |
|--------|-------------|---------------|
| **R / G / B** | Per-channel color balance | 0.5 - 1.5 |
| **Saturation** | Color intensity | 1.0 - 2.5 (SHO: >1.5, RGB: ~1.2) |
| **SCNR** | Reduces green excess (SHO) | 0.5=aggressive, 1.0+=disabled |
| **Extra gamma** | Brightens midtones | 1.0=none, 1.5=moderate |
| **Denoise** | Smooths noise | 0=off, 5=normal, 9=strong |

### Common mistakes:

- **Saturation < 1.0**: REDUCES color (everything gray). Always &ge; 1.0
- **SCNR too low**: removes all green &rarr; purple image
- **Gamma too high**: washed-out image, loses contrast

In [ ]:
# Prepare preview for post-processing
preview_pre = cv2.resize(final_pre, None, fx=PREVIEW_SCALE, fy=PREVIEW_SCALE,
                         interpolation=cv2.INTER_AREA)

# Defaults per mode
if MODE == 'SHO':
    def_R, def_G, def_B = 1.0, 0.75, 1.1
    def_sat = 1.8
    def_scnr = 0.85
else:
    def_R, def_G, def_B = 1.0, 1.0, 1.0
    def_sat = 1.2
    def_scnr = 1.1  # disabled for RGB

# Post-processing sliders
pp = {}
pp['R']       = widgets.FloatSlider(value=def_R, min=0.3, max=1.8, step=0.05,
                    description='Red:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['G']       = widgets.FloatSlider(value=def_G, min=0.3, max=1.8, step=0.05,
                    description='Green:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['B']       = widgets.FloatSlider(value=def_B, min=0.3, max=1.8, step=0.05,
                    description='Blue:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['sat']     = widgets.FloatSlider(value=def_sat, min=0.5, max=3.0, step=0.1,
                    description='Saturation:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['scnr']    = widgets.FloatSlider(value=def_scnr, min=0.3, max=1.5, step=0.05,
                    description='SCNR:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['gamma']   = widgets.FloatSlider(value=1.2, min=0.5, max=2.5, step=0.1,
                    description='Extra gamma:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))
pp['denoise'] = widgets.IntSlider(value=5, min=0, max=11, step=2,
                    description='Denoise:', style={'description_width': '80px'},
                    layout=widgets.Layout(width='400px'))

out_pp = widgets.Output()

def apply_postprocess(img, R, G, B, sat, scnr, gamma, denoise):
    """Apply post-processing to an RGB image [0,1]."""
    proc = img.copy()
    # Color balance
    proc[:,:,0] *= R
    proc[:,:,1] *= G
    proc[:,:,2] *= B
    proc = np.clip(proc, 0, 1)
    # SCNR
    if scnr <= 1.0:
        r_c, g_c, b_c = proc[:,:,0], proc[:,:,1], proc[:,:,2]
        limit = (r_c + b_c) / 2.0 * scnr + g_c * (1 - scnr)
        proc[:,:,1] = np.minimum(g_c, limit)
        proc = np.clip(proc, 0, 1)
    # Saturation
    p8 = (proc * 255).astype(np.uint8)
    hsv = cv2.cvtColor(p8, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:,:,1] = np.clip(hsv[:,:,1] * sat, 0, 255)
    proc = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32) / 255.0
    # Gamma
    if gamma != 1.0:
        proc = np.power(np.clip(proc, 0, 1), 1.0 / gamma)
    # Background
    for ch in range(3):
        bg = np.percentile(proc[:,:,ch], 15)
        proc[:,:,ch] = np.clip(proc[:,:,ch] - bg * 0.8, 0, None)
    mx = proc.max()
    if mx > 0: proc = proc / mx
    # Denoise
    if denoise > 0:
        p8 = (np.clip(proc, 0, 1) * 255).astype(np.uint8)
        proc = cv2.bilateralFilter(p8, denoise, 30, 30).astype(np.float32) / 255.0
    return proc

def update_postprocess(**kwargs):
    with out_pp:
        clear_output(wait=True)
        result = apply_postprocess(
            preview_pre,
            pp['R'].value, pp['G'].value, pp['B'].value,
            pp['sat'].value, pp['scnr'].value, pp['gamma'].value, pp['denoise'].value
        )
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
        axes[0].imshow(preview_pre, origin='lower')
        axes[0].set_title('Original', fontsize=11); axes[0].axis('off')
        axes[1].imshow(result, origin='lower')
        axes[1].set_title('Post-processed', fontsize=11); axes[1].axis('off')
        plt.suptitle(f'{OBJECT} - {MODE} Post-processing', fontsize=13)
        plt.tight_layout(); plt.show()

for ctrl in pp.values():
    ctrl.observe(lambda c: update_postprocess(), names='value')

col1 = widgets.VBox([
    widgets.HTML('<b>Color balance</b>'),
    pp['R'], pp['G'], pp['B'],
])
col2 = widgets.VBox([
    widgets.HTML('<b>Global adjustments</b>'),
    pp['sat'], pp['scnr'], pp['gamma'], pp['denoise'],
])

display(widgets.VBox([
    widgets.HTML(f'<h3>Post-processing: {MODE} ({OBJECT})</h3>'),
    widgets.HBox([col1, widgets.HTML('&nbsp;&nbsp;&nbsp;'), col2]),
]))
display(out_pp)
update_postprocess()

## Step 13: Apply post-processing at full resolution

When you are satisfied with the preview, run this cell.
It applies exactly the same parameters to the 61 MP image.

In [ ]:
print("Applying post-processing at full resolution...\n")
print(f"  R={pp['R'].value}, G={pp['G'].value}, B={pp['B'].value}")
print(f"  Sat={pp['sat'].value}, SCNR={pp['scnr'].value}")
print(f"  Gamma={pp['gamma'].value}, Denoise={pp['denoise'].value}")

final_proc = apply_postprocess(
    final_pre,
    pp['R'].value, pp['G'].value, pp['B'].value,
    pp['sat'].value, pp['scnr'].value, pp['gamma'].value, pp['denoise'].value
)

print("\nPost-processing applied at full resolution.")

## Step 14: Export

- **PNG** (8-bit): for viewing and sharing
- **TIFF** (16-bit): for additional post-processing in PixInsight or Photoshop

In [ ]:
obj_clean = OBJECT.replace(' ', '_')
prefix = f"{obj_clean}_{MODE}_final"

png_path = OUTPUT_DIR / f'{prefix}.png'
img8 = (np.clip(final_proc, 0, 1) * 255).astype(np.uint8)
Image.fromarray(np.flipud(img8)).save(str(png_path), 'PNG')
print(f"  PNG:  {png_path.name}  ({os.path.getsize(str(png_path))/1e6:.1f} MB)")

tiff_path = OUTPUT_DIR / f'{prefix}.tiff'
img16 = (np.clip(final_proc, 0, 1) * 65535).astype(np.uint16)
cv2.imwrite(str(tiff_path), np.flipud(img16[:,:,::-1]))
print(f"  TIFF: {tiff_path.name}  ({os.path.getsize(str(tiff_path))/1e6:.1f} MB)")

fig, ax = plt.subplots(figsize=(14, 9))
ax.imshow(np.flipud(img8))
ax.set_title(f'{OBJECT} - {MODE} Final', fontsize=14); ax.axis('off')
plt.tight_layout(); plt.show()

print(f"\nFiles in: {OUTPUT_DIR.absolute()}")

## Quick reference

### Complete pipeline:

```
  Raw FITS
     |
  [1] Calibration: (light - (dark - pedestal)) / flat
     |
  [2] Intra-filter alignment (phase correlation)
     |
  [3] Stacking (median)
     |
  [4] Inter-filter alignment (interactive sliders)
     |
  [5] Composition (SHO or RGB) + stretch
     |
  [6] LRGB (optional)
     |
  [7] Post-processing (interactive sliders)
     |
  Final PNG + TIFF
```

### Parameters by mode:

| Parameter | SHO | RGB | HaRGB / HaOIII-RGB |
|-----------|-----|-----|---------------------|
| STRETCH | 0.05-0.1 | 0.2-0.5 | 0.2-0.5 |
| GAMMA | 3.0-4.0 | 2.0-3.0 | 2.0-3.0 |
| SAT_FACTOR | 1.5-2.5 | 1.0-1.5 | 1.2-1.8 |
| SCNR | 0.7-0.9 | >1.0 (off) | >1.0 (off) |
| G_FACTOR | 0.7-0.8 | 1.0 | 1.0 |
| NB_MIX | n/a | n/a | 0.2-0.5 |

### Common issues:

| Problem | Solution |
|---------|----------|
| Image too dark | Lower STRETCH, raise GAMMA or Extra gamma |
| Image too green | Lower G in post-processing, lower SCNR |
| Purple image | Raise SCNR (to 0.95+), lower B |
| Weak/gray colors | Raise Saturation (always &ge; 1.0!) |
| Stars with fringes | Adjust alignment sliders |
| Too much noise | Raise Denoise (7-9) or capture more frames |

### To process another object:

1. Go back to **Step 4** and select another object in the dropdown
2. Select the desired mode (the dropdown updates automatically)
3. Run all cells again (Kernel &rarr; Restart & Run All)

### To improve the capture:

- More frames per filter (10-20) dramatically improves SNR
- More luminance exposure (5-10 &times; 300s) for effective LRGB
- Dithering between frames eliminates sensor patterns
- Quality is proportional to **total integration time**

### If sliders don't appear:

```python
!pip install ipywidgets ipympl
!jupyter nbextension enable --py widgetsnbextension
```

If using JupyterLab: `!jupyter labextension install @jupyter-widgets/jupyterlab-manager`

If `%matplotlib widget` errors out, replace it with `%matplotlib inline` in the imports cell.